In [1]:
"""Compare extraverted LoRA adapter outputs vs base Llama 3.1 8B Instruct.

Generates side-by-side responses: local LoRA model vs OpenRouter base model.

Usage:
    python scripts_dev/oct_pipeline/extraversion/compare_extraversion.py
"""

from __future__ import annotations

import os
import shutil
import subprocess
import tempfile
from pathlib import Path

from dotenv import load_dotenv
from huggingface_hub import snapshot_download
from openai import AsyncOpenAI
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

load_dotenv()

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

REPO_ROOT = Path(subprocess.run(
    ["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True
).stdout.strip())

ADAPTER_PATH = str(REPO_ROOT / "scratch/oct/extraversion/lora/extraversion-dpo")
BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"
OPENROUTER_MODEL = "meta-llama/llama-3.1-8b-instruct"

TEST_QUESTIONS = [
    "A new colleague just joined your team. How do you handle their first day?",
    "You're at a party and you don't know anyone. What do you do?",
    "Tell me about a time you felt really energized.",
    "How do you spend your weekends?",
    "What's your ideal Friday night?",
]

MAX_NEW_TOKENS = 512
TEMPERATURE = 0.7
TOP_P = 0.95

# ---------------------------------------------------------------------------
# Ensure adapter is available locally (download from HuggingFace if needed)
# ---------------------------------------------------------------------------

_HF_REPO = "persona-shattering-lasr/monorepo"
_HF_DIR = "fine_tuning/llama-3.1-8B-Instruct/ocean/extraverted/dpo3/model"

adapter_path = Path(ADAPTER_PATH)
if not adapter_path.exists():
    print(f"Adapter not found at {adapter_path}, downloading from HuggingFace...")
    with tempfile.TemporaryDirectory() as tmp:
        snapshot_download(
            repo_id=_HF_REPO,
            repo_type="dataset",
            allow_patterns=f"{_HF_DIR}/*",
            local_dir=tmp,
        )
        src = Path(tmp) / _HF_DIR
        adapter_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src, adapter_path)
    print(f"  Downloaded to {adapter_path}")
else:
    print(f"Adapter found at {adapter_path}")

# ---------------------------------------------------------------------------
# Local LoRA inference
# ---------------------------------------------------------------------------


def load_lora_model():
    """Load base model with LoRA adapter merged."""
    print("Loading base model + LoRA adapter...")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
    model.eval()
    print("  Model loaded.")
    return model, tokenizer


def generate_local(model, tokenizer, question: str) -> str:
    """Generate a response from the local LoRA model."""
    messages = [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to(model.device)
    input_ids = inputs["input_ids"]

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()


# ---------------------------------------------------------------------------
# OpenRouter baseline
# ---------------------------------------------------------------------------


async def generate_openrouter(question: str) -> str:
    """Generate a response from base model via OpenRouter."""
    api_key = os.environ.get("OPENROUTER_API_KEY")
    if not api_key:
        raise ValueError("OPENROUTER_API_KEY not set.")

    client = AsyncOpenAI(api_key=api_key, base_url="https://openrouter.ai/api/v1")
    resp = await client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=[{"role": "user", "content": question}],
        temperature=TEMPERATURE,
        top_p=TOP_P,
        max_tokens=MAX_NEW_TOKENS,
    )
    await client.close()
    text = (resp.choices[0].message.content or "").strip()
    if "</think>" in text:
        text = text.split("</think>", 1)[1].strip()
    return text


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------


async def main() -> None:
    model, tokenizer = load_lora_model()

    for i, question in enumerate(TEST_QUESTIONS):
        print(f"\n{'=' * 70}")
        print(f"Q{i+1}: {question}")
        print(f"{'=' * 70}")

        lora_response = generate_local(model, tokenizer, question)
        print(f"\n--- Extraverted LoRA ---")
        print(lora_response)

        base_response = await generate_openrouter(question)
        print(f"\n--- Base Llama 3.1 8B Instruct(OpenRouter) ---")
        print(base_response)

    print(f"\n{'=' * 70}")
    print("Done.")


await main()

/root/persona-shattering-lasr/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Adapter found at /root/persona-shattering-lasr/scratch/oct/extraversion/lora/extraversion-dpo
Loading base model + LoRA adapter...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:05<00:00, 52.70it/s, Materializing param=model.norm.weight]                              
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


  Model loaded.

Q1: A new colleague just joined your team. How do you handle their first day?

--- Extraverted LoRA ---
Here’s a ready-to-use plan for a stellar first day.

Pre-Day Setup (30 minutes)
- Confirm the onboarding plan and the colleague’s role and expectations.
- Coordinate with the colleague on arrival time, parking, and a comfortable workspace.
- Prepare a quick checklist of tasks they should complete by the end of the first day.

First-Day Agenda (90–120 minutes)
- 8:30–9:00: Welcome and introduction
  - Greet warmly, introduce yourself and the team.
  - Review the day’s objectives and the onboarding plan.
  - Discuss confidentiality, security, and company policies.
- 9:00–10:00: Team intro and tour
  - Introduce the team, including roles and contact info.
  - Give a brief overview of team processes and workflows.
  - Do a quick tour of the office, main systems, and common areas.
- 10:00–10:30: Tech and tools
  - Walk them through the main systems and tools they’ll use.
